In [0]:
df_medications = spark.sql(f"select * from regis_healthcare.silver.medications;")
df_medications.createOrReplaceTempView("medications")

In [0]:
# 5. Dim_Medication -- >Source: medications
# | Column             |
# | ------------------ |
# | medication_key     |
# | medication_id      |
# | medication_name    |
# | dosage             |
# | frequency          |
# | prescribing_doctor |
dim_medication = spark.sql("""select * from medications order by medication_id""")
# display(Dim_Medication)
from pyspark.sql.functions import col, regexp_replace
dim_medication = dim_medication.withColumn(
    "medication_key",
    regexp_replace(col("medication_id"), "^MED", "").cast("int")
)
# display(df_medicationss)
dim_medication = dim_medication.select(
  "medication_key",     
  "medication_id",      
  "medication_name",    
  "dosage",             
  "frequency",         
  "prescribing_doctor" 
)
display(dim_medication)

#### cataloge 

In [0]:
dim_medication.write\
    .format("delta")\
    .option("mergeSchema","true")\
    .option("overwriteSchema","true")\
        .option("delta.enableChangeDataFeed","true")\
            .mode("overwrite")\
.saveAsTable(f"regis_healthcare.gold.dim_medication")

In [0]:
dim_medication.write\
    .format("delta")\
    .option("mergeSchema","true")\
    .option("overwriteSchema","true")\
        .option("delta.enableChangeDataFeed","true")\
            .mode("overwrite")\
.saveAsTable(f"regis_healthcare.gold.sb_dim_medication")
print(dim_medication.count())

In [0]:
from delta.tables import DeltaTable
from pyspark.sql import functions as F

# Load Delta table with correct fully-qualified name
delta_table = DeltaTable.forName(spark, "regis_healthcare.gold.dim_medication")
# Create DataFrame from source table with correct fully-qualified name
# sb_dim_products
df_child_products = (
    spark.table("regis_healthcare.gold.sb_dim_medication")
    .select("*")
)
# Perform merge
delta_table.alias("target").merge(
    source=df_child_products.alias("source"),
    condition="target.medication_key = source.medication_key"
).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

In [0]:
dim_df = spark.sql(f"select * from regis_healthcare.gold.dim_medication;")
print(dim_df.count())

sb_dim_df = spark.sql(f"select * from regis_healthcare.gold.sb_dim_facilities;")
print(sb_dim_df.count())

#### s3 loading

In [0]:
# gold load to s3
dim_medication.write\
    .format("delta")\
    .option("mergeSchema","true")\
    .option("overwriteSchema","true")\
        .option("delta.enableChangeDataFeed","true")\
            .mode("overwrite")\
.save(f"s3://regis-healthcare/gold-delta-table/dim_medication")

In [0]:
from delta.tables import DeltaTable

# ✅ Path to your Delta table stored in S3
delta_table_path = f"s3://regis-healthcare/gold-delta-table/dim_medication"

# ✅ Load target Delta table
delta_table = DeltaTable.forPath(spark, delta_table_path)

# ✅ Source DataFrame (example: df_child_products)
source_df = dim_medication

# ✅ Perform MERGE with upsert logic
(
    delta_table.alias("target")
    .merge(
        source_df.alias("source"),
        "target.medication_key = source.medication_key"
    )
    .whenMatchedUpdateAll()      # Update all columns when matched
    .whenNotMatchedInsertAll()   # Insert all columns when not matched
    .execute()
)
